# Research: more data, and stacking, for the average-player model

An independent experiment. It does **not** change the main rankings.

Three questions:
1. **Stacking.** Does combining several models (LightGBM, XGBoost, ExtraTrees, logistic regression) give a better
   "average player" than LightGBM alone?
2. **More data for the average player.** If the model also learns from ~5 lakh outside balls (Cricsheet), does it
   predict new balls better?
3. **More data for batter ratings.** If batters are rated on RR's data *plus* outside matches, do the ratings predict
   how batters do in other outside matches better than ratings from RR's data alone?

Every comparison is made on matches the models never saw.

In [1]:
import sys, os, json, time, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, os.path.abspath(".."))
import numpy as np, pandas as pd, lightgbm as lgb, xgboost as xgb
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.preprocessing import SplineTransformer, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import log_loss
from rrlib.common import CFG, SEED, NJ, load, load_json, load_table, save_table, save_json, weighted_spearman, excluded_players
from rrlib import ratings as RL

T0 = time.time()
say = lambda *a: print(f"[{time.time() - T0:6.0f}s]", *a, flush=True)
FAM = load_json("feature_families")
EXO_B = ["wkts_last6", "wkts_last12", "wkts_last24", "balls_since_wkt", "pship_balls", "str_balls", "ns_balls", "free_hit", "rebowl", "bwl_balls"]
FEATS = FAM["A_core"] + FAM["A_derived"] + EXO_B + FAM["C_context"] + ["is_pace"]
LGBP = dict(CFG["models"]["lgb_common"], n_jobs=NJ, random_state=SEED)
NC = 8
CLS = ["dot", "1", "2", "3", "4", "6", "wicket", "wide/nb"]

S = load("features")
S["fold"] = S.match_id.map(load("folds").set_index("match_id").fold).values
yS = S.y_class.values
say("supplied balls", len(S), "matches", S.match_id.nunique(), "| features", len(FEATS))


def inner_split(mids_tr, frac=0.15, seed=SEED):
    u = np.unique(mids_tr); r = np.random.RandomState(seed).permutation(u)
    val = set(r[: int(len(u) * frac)])
    return np.array([m in val for m in mids_tr])


def fit_lgb(X, y, mids, cat=None):
    m = inner_split(mids)
    d1 = lgb.Dataset(X[~m], y[~m], categorical_feature=cat or "auto"); d2 = lgb.Dataset(X[m], y[m], categorical_feature=cat or "auto")
    return lgb.train(dict(LGBP, objective="multiclass", num_class=NC, learning_rate=0.06), d1, 3000, valid_sets=[d2],
                     callbacks=[lgb.early_stopping(50, verbose=False)])


def ece_by_class(y, P):
    return {c: abs((y == i).mean() - P[:, i].mean()) for i, c in enumerate(CLS)}

[     0s] supplied balls 200961 matches 867 | features 41


## 1. Stacking for the average-player model

Same 5 groups of hidden matches as the main pipeline. For each group, every model learns on the other 4 groups and
predicts the hidden one. Then:
- **average**: the plain average of the models' probabilities;
- **stacking**: a logistic regression learns how to weigh the models, again judged only on hidden matches.

In [2]:
CORE_NUM = ["innings", "legal_before", "balls_left", "wkts_before", "score_before", "lvl_code", "year", "venue_par"]
Xl = S[CORE_NUM].copy(); Xl["need"] = S.need.fillna(0); Xl["rrr"] = S.rrr.fillna(0).clip(0, 36)
OOF = {k: np.zeros((len(S), NC)) for k in ("LightGBM", "XGBoost", "ExtraTrees", "Logistic (curves)")}
TRAIN_LL = {k: [] for k in OOF}
for k in range(5):
    tr, te = (S.fold != k).values, (S.fold == k).values
    Xtr, Xte, ytr, mtr = S.loc[tr, FEATS], S.loc[te, FEATS], yS[tr], S.match_id.values[tr]
    b = fit_lgb(Xtr, ytr, mtr)
    OOF["LightGBM"][te] = b.predict(Xte, num_iteration=b.best_iteration)
    TRAIN_LL["LightGBM"].append(log_loss(ytr, b.predict(Xtr, num_iteration=b.best_iteration), labels=range(NC)))
    m = inner_split(mtr)
    xm = xgb.XGBClassifier(objective="multi:softprob", num_class=NC, n_estimators=3000, learning_rate=0.06, max_depth=6,
                           min_child_weight=50, subsample=0.8, colsample_bytree=0.8, reg_lambda=5.0, tree_method="hist",
                           n_jobs=NJ, random_state=SEED, early_stopping_rounds=50, eval_metric="mlogloss")
    xm.fit(Xtr[~m], ytr[~m], eval_set=[(Xtr[m], ytr[m])], verbose=False)
    OOF["XGBoost"][te] = xm.predict_proba(Xte)
    TRAIN_LL["XGBoost"].append(log_loss(ytr, xm.predict_proba(Xtr), labels=range(NC)))
    et = ExtraTreesClassifier(n_estimators=200, min_samples_leaf=100, max_features=0.5, n_jobs=NJ, random_state=SEED)
    et.fit(Xtr.fillna(-1), ytr)
    OOF["ExtraTrees"][te] = np.clip(et.predict_proba(Xte.fillna(-1)), 1e-6, 1)
    TRAIN_LL["ExtraTrees"].append(log_loss(ytr, np.clip(et.predict_proba(Xtr.fillna(-1)), 1e-6, 1), labels=range(NC)))
    sub = np.random.RandomState(SEED).choice(np.where(tr)[0], min(60000, tr.sum()), replace=False)
    lr = make_pipeline(StandardScaler(), SplineTransformer(n_knots=5, degree=3), LogisticRegression(max_iter=400, C=1.0))
    lr.fit(Xl.iloc[sub], yS[sub])
    pr = np.zeros((te.sum(), NC)); pr[:, lr[-1].classes_] = lr.predict_proba(Xl[te]); OOF["Logistic (curves)"][te] = np.clip(pr, 1e-6, 1)
    TRAIN_LL["Logistic (curves)"].append(log_loss(yS[sub], np.clip(lr.predict_proba(Xl.iloc[sub]), 1e-6, 1), labels=range(NC)))
    say(f"fold {k} done")
for k_ in OOF:
    OOF[k_] = OOF[k_] / OOF[k_].sum(1, keepdims=True)

# simple averages
OOF["Average: LightGBM + XGBoost"] = (OOF["LightGBM"] + OOF["XGBoost"]) / 2
OOF["Average: all four"] = sum(OOF[k] for k in ("LightGBM", "XGBoost", "ExtraTrees", "Logistic (curves)")) / 4
# stacking: meta logistic regression on the log-probabilities of the four models, fitted on 4 groups, judged on the 5th
Z = np.hstack([np.log(np.clip(OOF[k], 1e-6, 1)) for k in ("LightGBM", "XGBoost", "ExtraTrees", "Logistic (curves)")])
st = np.zeros((len(S), NC))
for k in range(5):
    tr, te = (S.fold != k).values, (S.fold == k).values
    meta = LogisticRegression(max_iter=500, C=1.0).fit(Z[tr], yS[tr])
    st[te] = meta.predict_proba(Z[te])
OOF["Stacking (logistic on top of the four)"] = st
prior = np.bincount(yS, minlength=NC) / len(yS)
rows = [{"model": "No model (overall frequencies)", "unseen_logloss": log_loss(yS, np.tile(prior, (len(S), 1)), labels=range(NC)),
         "train_logloss": np.nan, "worst_class_rate_error_pct": 0.0, "top1_hit": (yS == prior.argmax()).mean()}]
for k_, P in OOF.items():
    rows.append({"model": k_, "unseen_logloss": log_loss(yS, P, labels=range(NC)),
                 "train_logloss": np.mean(TRAIN_LL[k_]) if k_ in TRAIN_LL else np.nan,
                 "worst_class_rate_error_pct": 100 * max(ece_by_class(yS, P).values()), "top1_hit": (P.argmax(1) == yS).mean()})
STACK = pd.DataFrame(rows)
STACK["gap"] = STACK.unseen_logloss - STACK.train_logloss
save_table(STACK, "research_stacking_average_player")
say("STACKING\n" + STACK.round(4).to_string())

[    72s] fold 0 done


[   143s] fold 1 done


[   224s] fold 2 done


[   301s] fold 3 done


[   377s] fold 4 done


[   621s] STACKING
                                    model  unseen_logloss  train_logloss  worst_class_rate_error_pct  top1_hit     gap
0          No model (overall frequencies)          1.6366            NaN                      0.0000    0.3574     NaN
1                                LightGBM          1.5781         1.5460                      0.0416    0.4125  0.0321
2                                 XGBoost          1.5783         1.5403                      0.1192    0.4128  0.0380
3                              ExtraTrees          1.5812         1.5193                      0.0469    0.4120  0.0619
4                       Logistic (curves)          1.5939         1.5881                      0.0514    0.4084  0.0058
5             Average: LightGBM + XGBoost          1.5776            NaN                      0.0537    0.4128     NaN
6                       Average: all four          1.5783            NaN                      0.0461    0.4124     NaN
7  Stacking (logistic on top 

## 2. More data for the average-player model

The outside matches are split by match, within each competition and season: about 55% for learning (~5 lakh balls)
and 45% for testing (~4 lakh balls). Two models:
- **RR data only** (the model the main system uses);
- **RR data + outside learning half**.

Both are judged on (a) the outside testing half, and (b) RR's own hidden matches.
Outside franchise leagues get their own competition code, so the model doesn't treat them as the IPL.

In [3]:
E = load("features_ext")
VE = load("values_ext", columns=["match_id", "comp_key"]).drop_duplicates("match_id")
E = E[E.match_id.isin(VE.match_id)].reset_index(drop=True)          # duplicates already removed in the outside test
ck = E.match_id.map(VE.set_index("match_id").comp_key)


def split_ext(seed, frac=0.55):
    """split outside matches by match, inside each competition-season"""
    rs = np.random.RandomState(seed); train = set()
    for _, g in VE.groupby("comp_key"):
        ids = g.match_id.values.copy(); rs.shuffle(ids); train |= set(ids[: int(round(len(ids) * frac))])
    return train


A = split_ext(SEED)
inA = E.match_id.isin(A).values
say("outside balls", len(E), "| learning half", inA.sum(), "| testing half", (~inA).sum())
LV2 = {"BBL": 4, "PSL": 4, "CPL": 4, "SA20": 4, "ILT20": 4, "MLC": 4, "BPL": 4, "LPL": 4, "Blast": 4, "SMAT": 5}
E2 = E.copy(); E2["lvl_code"] = E2.level.map(LV2).fillna(E2.lvl_code).astype(int)
yE = E.y_class.values

# (a) judged on the outside testing half
m_S = fit_lgb(S[FEATS], yS, S.match_id.values)
P_S = m_S.predict(E.loc[~inA, FEATS], num_iteration=m_S.best_iteration)
XSE = pd.concat([S[FEATS], E2.loc[inA, FEATS]], ignore_index=True)
ySE = np.concatenate([yS, yE[inA]]); mSE = np.concatenate([S.match_id.values, E2.match_id.values[inA]])
m_SE = fit_lgb(XSE, ySE, mSE)
P_SE = m_SE.predict(E2.loc[~inA, FEATS], num_iteration=m_SE.best_iteration)
rowsB = []
lvl_te = E.level.values[~inA]
for name, P in (("RR data only", P_S), ("RR data + outside learning half", P_SE)):
    rowsB.append({"model": name, "judged_on": "outside testing half (all)", "logloss": log_loss(yE[~inA], P, labels=range(NC)),
                  "worst_class_rate_error_pct": 100 * max(ece_by_class(yE[~inA], P).values())})
    for grp, msk in (("associate T20Is", lvl_te == "T20I-assoc"), ("franchise leagues", np.isin(lvl_te, list(k for k, v_ in LV2.items() if v_ == 4)))):
        rowsB.append({"model": name, "judged_on": f"outside testing half ({grp})", "logloss": log_loss(yE[~inA][msk], P[msk], labels=range(NC)),
                      "worst_class_rate_error_pct": 100 * max(ece_by_class(yE[~inA][msk], P[msk]).values())})
say("(a) done")

# (b) judged on RR's own hidden matches (same 5 groups as the main pipeline)
oof_SE = np.zeros((len(S), NC))
for k in range(5):
    tr, te = (S.fold != k).values, (S.fold == k).values
    X_ = pd.concat([S.loc[tr, FEATS], E2.loc[inA, FEATS]], ignore_index=True)
    y_ = np.concatenate([yS[tr], yE[inA]]); m_ = np.concatenate([S.match_id.values[tr], E2.match_id.values[inA]])
    mk = fit_lgb(X_, y_, m_)
    oof_SE[te] = mk.predict(S.loc[te, FEATS], num_iteration=mk.best_iteration)
    say(f"(b) fold {k} done")
for name, P in (("RR data only", OOF["LightGBM"]), ("RR data + outside learning half", oof_SE)):
    rowsB.append({"model": name, "judged_on": "RR's own hidden matches", "logloss": log_loss(yS, P, labels=range(NC)),
                  "worst_class_rate_error_pct": 100 * max(ece_by_class(yS, P).values())})
MORE_AVG = pd.DataFrame(rowsB)
save_table(MORE_AVG, "research_more_data_average_player")
say("MORE DATA, AVERAGE PLAYER\n" + MORE_AVG.round(4).to_string())

[   628s] outside balls 907349 | learning half 499290 | testing half 408059


[   692s] (a) done


[   732s] (b) fold 0 done


[   773s] (b) fold 1 done


[   805s] (b) fold 2 done


[   837s] (b) fold 3 done


[   865s] (b) fold 4 done


[   866s] MORE DATA, AVERAGE PLAYER
                             model                                 judged_on  logloss  worst_class_rate_error_pct
0                     RR data only                outside testing half (all)   1.5653                      2.4385
1                     RR data only    outside testing half (associate T20Is)   1.5650                      4.5311
2                     RR data only  outside testing half (franchise leagues)   1.5687                      1.1203
3  RR data + outside learning half                outside testing half (all)   1.5474                      0.1785
4  RR data + outside learning half    outside testing half (associate T20Is)   1.5365                      0.2555
5  RR data + outside learning half  outside testing half (franchise leagues)   1.5601                      0.2606
6                     RR data only                   RR's own hidden matches   1.5781                      0.0416
7  RR data + outside learning half                  

## 3. More data for batter ratings

Batters are rated exactly as in the main system (pressure-weighted, pulled towards average, at least 60 pressure
balls), using:
- **RR data only**;
- **RR data + outside learning half**;
- **outside learning half only** (for reference).

Then each rating is checked against how the same batters did on pressure balls in the **outside testing half**:
runs above an average batter per pressure ball. Only batters with at least 20 pressure balls there are judged.
Repeated on 5 random splits and one forward-in-time split (learn on outside 2022 to 2024, test on 2025 to 2026).

In [4]:
V0 = load("values")
VX = load("values_ext")
comp = load_json("p05_components")
chosen = load_json("p09_summary")["chosen"]
players = load("players").set_index("player_id")
EXCL = excluded_players()
keep = ["ball_uid", "match_id", "year", "level", "striker_id", "bowler_id", "is_pressure", "pressure_pct", "faced", "runs", "team_runs",
        "legal", "phase", "is_pace"] + [c for c in V0.columns if c[:3] in ("v1_", "v2_", "v3_", "v4_")] + sum(RL.COMPS.values(), [])
keep = [c for c in dict.fromkeys(keep) if c in V0.columns and c in VX.columns]
pc = RL.PCOL["bat"]
sc_S = RL.build_scores(V0, "bat", chosen["bat"], comp).score
base_ids = set(V0.loc[V0.is_pressure & V0.v1_bat.notna(), pc].unique())
splits = {f"random split {i + 1}": split_ext(SEED + i) for i in range(5)}
splits["forward in time (learn 2022-24, test 2025-26)"] = set(VX.loc[VX.year <= 2024, "match_id"].unique())
rowsC = []
for sname, Aset in splits.items():
    a_ = VX.match_id.isin(Aset)
    sc_SE = RL.build_scores(pd.concat([V0[keep], VX.loc[a_, keep]], ignore_index=True), "bat", chosen["bat"], comp).score
    sc_E = RL.build_scores(VX.loc[a_, keep].reset_index(drop=True), "bat", chosen["bat"], comp).score
    d = VX[~a_ & VX.is_pressure & VX.wpa_bat.notna()]
    y = d.groupby(pc).agg(runs=("v1_bat", "mean"), wp=("wpa_bat", "mean"), n=("v1_bat", "size"))
    y = y[y.n >= 20]
    pools = {"RR's batters (all three ratings can judge them)": y[y.index.isin(players.index) & y.index.isin(base_ids) & ~y.index.isin(EXCL)]}
    for pool_name, yy in pools.items():
        for mname, sc in (("RR data only", sc_S), ("RR data + outside learning half", sc_SE), ("outside learning half only", sc_E)):
            ss = sc.reindex(yy.index).fillna(-1e9)
            top = ss.sort_values(ascending=False).index[:10]
            rowsC.append({"split": sname, "pool": pool_name, "rating": mname, "batters_judged": len(yy),
                          "rank_corr_runs": weighted_spearman(ss.values, yy.runs.values, yy.n.values),
                          "rank_corr_winchance": weighted_spearman(ss.values, yy.wp.values, yy.n.values),
                          "top10_extra_runs_per_100_pressure_balls": 100 * (np.average(yy.loc[top, "runs"], weights=yy.loc[top, "n"]) - np.average(yy.runs, weights=yy.n)),
                          "batters_rated": int(sc.notna().sum())})
    say(f"{sname} done")
RAT = pd.DataFrame(rowsC)
save_table(RAT, "research_more_data_batter_ratings")
SUMM = RAT.groupby("rating")[["rank_corr_runs", "rank_corr_winchance", "top10_extra_runs_per_100_pressure_balls"]].agg(["mean", "min", "max"])
say("MORE DATA, BATTER RATINGS (per split)\n" + RAT.drop(columns="pool").round(3).to_string())
say("summary over the 6 splits\n" + SUMM.round(3).to_string())

# how the batting top 10 would change with all outside matches added (descriptive only)
sc_all = RL.build_scores(pd.concat([V0[keep], VX[keep]], ignore_index=True), "bat", chosen["bat"], comp)
sc_all = sc_all[sc_all.index.isin(players.index) & ~sc_all.index.isin(EXCL)].sort_values("score", ascending=False)
main10 = list(load_table("final_top30_bat").head(10).player_id)
new10 = list(sc_all.index[:10])
save_json({"main_top10": [players.name.get(p, p) for p in main10], "with_all_outside_top10": [players.name.get(p, p) for p in new10],
           "shared": len(set(main10) & set(new10))}, "research_more_data_top10_change")
say("top 10 with all outside matches added:", [players.name.get(p, p) for p in new10], "| shared with main:", len(set(main10) & set(new10)))
say("finished")

[   872s] random split 1 done


[   876s] random split 2 done


[   879s] random split 3 done


[   883s] random split 4 done


[   887s] random split 5 done


[   891s] forward in time (learn 2022-24, test 2025-26) done


[   891s] MORE DATA, BATTER RATINGS (per split)
                                            split                           rating  batters_judged  rank_corr_runs  rank_corr_winchance  top10_extra_runs_per_100_pressure_balls  batters_rated
0                                  random split 1                     RR data only             172           0.118                0.141                                    0.852            176
1                                  random split 1  RR data + outside learning half             172           0.030                0.051                                   -4.926            491
2                                  random split 1       outside learning half only             172          -0.033               -0.056                                    0.061            375
3                                  random split 2                     RR data only             169           0.005                0.089                                   -2.580        

[   891s] summary over the 6 splits
                                rank_corr_runs               rank_corr_winchance               top10_extra_runs_per_100_pressure_balls               
                                          mean    min    max                mean    min    max                                    mean    min     max
rating                                                                                                                                               
RR data + outside learning half          0.060  0.006  0.133               0.110  0.006  0.213                                  -0.986 -9.887   9.506
RR data only                             0.051  0.005  0.118               0.086  0.015  0.145                                   3.513 -4.786  11.091
outside learning half only              -0.013 -0.118  0.046               0.019 -0.068  0.097                                  -0.468 -5.348   2.774


[   894s] top 10 with all outside matches added: ['AHT Donald', 'Vaibhav Sooryavanshi', 'Heinrich Klaasen', 'Abhishek Sharma', 'Josh Inglis', 'Suryakumar Yadav', 'Finn Allen', 'Tim David', 'Rovman Powell', 'Rinku Singh'] | shared with main: 7


[   894s] finished
